# AD vs FTD: C3 on both recordings of each patient

Experiment 10 confirmed C3 (lagged cross-correlation) on ds006036 (AUC 0.641, p 0.045). Each patient was recorded
twice: eyes closed (ds004504, 300 s) and eyes open under photic stimulation (ds006036, 44 s). The two C3 scores
agree only partly (Spearman r 0.40), so averaging them may cancel some noise.

- **One rule, fixed in Cell 2** (`results/c3_both/protocol.json`) before the average is computed. The score is the
  plain mean of the two out-of-fold C3 probabilities, and no other way of combining them is tried.
- **Already seen:** both inputs (0.628 and 0.641). The ds004504 half is the recording on which C3 was picked, so
  the combined AUC is somewhat optimistic; Experiment 10's 0.641 stays the clean number.
- **Nothing is retrained** for the headline. Cell 4 reproduces both inputs exactly from the saved matrices, and
  the permutation test re-runs both pipelines on the same shuffled diagnoses.
- **Reads** `features/riemann_matrices.npz`, `features/c3_ds006036_matrices.npz`, `results/riemann/`,
  `results/c3_ds006036/`, `../subject_cache/` and `../results_gat_multiclass/` (3-class check).
- **Writes** `results/c3_both/`. About 30 min, almost all of it the permutation test (CPU).

In [1]:
# ===========================================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ===========================================================================
import json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, recall_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")

EXP_DIR  = Path.cwd()                         # experiments_ad_vs_ftd/
PROJECT  = EXP_DIR.parent
FEAT_DIR = EXP_DIR / "features"
OUT_DIR  = EXP_DIR / "results" / "c3_both"
OUT_DIR.mkdir(parents=True, exist_ok=True)
HC, AD, FTD = 0, 1, 2

SEED, N_FOLDS, N_REPEATS = 42, 5, 5   # Experiment 9's folds and seeds
INNER_FOLDS = 5
C_GRID      = [0.001, 0.01, 0.1, 1.0]
N_BOOT      = 2000
N_PERM      = 1000
N_JOBS      = max(1, min(8, (os.cpu_count() or 2) - 2))
print(f"output: {OUT_DIR}\nworkers: {N_JOBS}")


output: C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_both
workers: 8


In [2]:
# ===========================================================================
# CELL 2 — PROTOCOL, WRITTEN BEFORE THE AVERAGE IS COMPUTED
# ===========================================================================
PROTOCOL = {
    "question": "Does averaging each patient's two C3 scores (ds004504 eyes closed, 300 s; ds006036 eyes open with "
                "photic stimulation, 44 s) separate AD (36) from FTD (23) better than either recording alone?",
    "score": "the plain mean of the two out-of-fold C3 probabilities q_AD (Experiment 9's C3 and Experiment 10's "
             "primary run, same folds and seeds); no other combination is tried",
    "already_seen": "both inputs: ds004504 C3 0.628 (picked after the fact in Experiment 9) and ds006036 C3 0.641 "
                    "(Experiment 10). The combined AUC inherits the ds004504 half's selection optimism",
    "primary": "AUC of the averaged score, AD vs FTD on the 59 patients, mean over 5 repeats, 95 % CI from 2000 "
               "stratified subject bootstraps; one-sided permutation p from 1000 shuffles of the diagnoses run "
               "through both pipelines and averaged",
    "secondary": {
        "gain": "paired bootstrap of the AUC difference vs each recording alone",
        "severity": "likelihood-ratio test of the averaged score added to MMSE + age + sex",
        "three_class": "two-stage 3-class model: stage 1 = P(dementia) from the GAT + band-power soft vote "
                       "(results_gat_multiclass/ensemble_oof_prob.npy, repeats 1-3, identical folds); P(HC) = "
                       "1 - p_d, P(AD) = p_d q, P(FTD) = p_d (1 - q); accuracy, macro-F1, macro recall, recall per class",
    },
    "target": "AD vs FTD AUC >= 0.70",
    "decision": "reported whatever it is. The averaged score replaces ds006036 C3 as stage 2 only if its lower 95 % "
                "bound of the gain over ds006036 alone is above 0",
}

_path = OUT_DIR / "protocol.json"
if _path.exists():
    _old = json.loads(_path.read_text(encoding="utf-8"))
    assert {k: v for k, v in _old.items() if k != "written"} == PROTOCOL, "protocol changed after it was written"
    PROTOCOL = _old
else:
    PROTOCOL = {"written": time.strftime("%Y-%m-%d %H:%M") + ", before the average was computed", **PROTOCOL}
    _path.write_text(json.dumps(PROTOCOL, indent=1, ensure_ascii=False), encoding="utf-8")
print(f"protocol: {_path}  (written {PROTOCOL['written']})")


protocol: C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_both\protocol.json  (written 2026-11-16 18:00, before the average was computed)


In [3]:
# ===========================================================================
# CELL 3 — SUBJECTS, FOLDS AND THE TWO SAVED INPUTS
# ===========================================================================
subjects = pd.read_csv(FEAT_DIR / "subjects.csv")
LABELS   = subjects.label.to_numpy()
DEMENTIA = np.flatnonzero(LABELS != HC)

_s2 = np.load(EXP_DIR / "results" / "riemann" / "stage2_oof.npz", allow_pickle=True)
_e10 = np.load(EXP_DIR / "results" / "c3_ds006036" / "c3_oof.npz", allow_pickle=True)
assert (_s2["fold_of"] == _e10["fold_of"]).all() and (_s2["labels"] == LABELS).all() and (_e10["labels"] == LABELS).all()
FOLD_OF = _s2["fold_of"]
FOLDS = [dict(repeat=r + 1, fold=f, test=np.flatnonzero(FOLD_OF[r] == f))
         for r in range(N_REPEATS) for f in range(1, N_FOLDS + 1)]
Q04_SAVED = _s2["q_ad"][list(_s2["candidates"]).index("C3 lagged cross-correlation")]
Q06_SAVED = _e10["q_ad"][list(_e10["runs"]).index("ds006036, 44 s (primary)")]

with np.load(FEAT_DIR / "riemann_matrices.npz") as z:
    X04 = z["xcz"]                                            # ds004504, 300 s
with np.load(FEAT_DIR / "c3_ds006036_matrices.npz") as z:
    X06 = z["xcz_06"]                                         # ds006036, 44 s
print(f"{len(subjects)} subjects, {len(FOLDS)} folds; matrices {X04.shape} and {X06.shape}")


88 subjects, 25 folds; matrices (88, 684) and (88, 684)


In [4]:
# ===========================================================================
# CELL 4 — THE C3 PIPELINE, CHECKED AGAINST BOTH SAVED INPUTS, AND THE AVERAGE
# ===========================================================================
def run_fold(X, test, labels, seed):
    """Experiment 9's C3: tuned on the dementia patients outside `test`, q_AD for every test patient."""
    train = np.setdiff1d(np.flatnonzero(labels != HC), test)
    inner = StratifiedKFold(INNER_FOLDS, shuffle=True, random_state=seed)
    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=0.1, class_weight="balanced", max_iter=5000))
    est = GridSearchCV(pipe, {"logisticregression__C": C_GRID}, cv=inner, scoring="roc_auc")
    return est.fit(X[train], (labels[train] == AD).astype(int)).predict_proba(X[test])[:, 1]


def run_cv(X, labels):
    q = np.full((N_REPEATS, len(labels)), np.nan)
    for s in FOLDS:
        q[s["repeat"] - 1, s["test"]] = run_fold(X, s["test"], labels, SEED + 100 * s["repeat"] + s["fold"])
    return q


assert np.allclose(run_cv(X04, LABELS), Q04_SAVED, atol=1e-10), "ds004504 C3 not reproduced"
assert np.allclose(run_cv(X06, LABELS), Q06_SAVED, atol=1e-10), "ds006036 C3 not reproduced"
print("identity checks passed: both saved C3 inputs reproduced exactly")
Q = {"ds004504 eyes closed, 300 s": Q04_SAVED,
     "ds006036 eyes open, 44 s (Experiment 10)": Q06_SAVED,
     "both recordings, averaged (primary)": (Q04_SAVED + Q06_SAVED) / 2}
PRIMARY = "both recordings, averaged (primary)"


identity checks passed: both saved C3 inputs reproduced exactly


In [5]:
# ===========================================================================
# CELL 5 — AD VS FTD: AUC, CIs AND THE GAIN OVER EACH RECORDING
# ===========================================================================
YD = (LABELS[DEMENTIA] == AD).astype(int)
POS, NEG = np.flatnonzero(YD == 1), np.flatnonzero(YD == 0)


def auc(pos_scores, neg_scores):
    d = pos_scores[..., :, None] - neg_scores[..., None, :]
    return (d > 0).mean((-1, -2)) + 0.5 * (d == 0).mean((-1, -2))


_rng = np.random.default_rng(SEED)
_bp, _bn = _rng.choice(POS, (N_BOOT, len(POS))), _rng.choice(NEG, (N_BOOT, len(NEG)))
BOOT, rows = {}, []
for n, q in Q.items():
    qd = q[:, DEMENTIA]
    BOOT[n] = np.stack([auc(qd[r][_bp], qd[r][_bn]) for r in range(N_REPEATS)]).mean(0)
    call = qd >= 0.5
    rows.append(dict(run=n, auc=auc(qd[:, POS], qd[:, NEG]).mean(), auc_lo=np.percentile(BOOT[n], 2.5),
                     auc_hi=np.percentile(BOOT[n], 97.5), recall_AD=call[:, POS].mean(),
                     recall_FTD=(~call[:, NEG]).mean()))
RESULTS = pd.DataFrame(rows).set_index("run")
RESULTS["bal_acc"] = (RESULTS.recall_AD + RESULTS.recall_FTD) / 2
GAIN = {}
for n in Q:
    if n != PRIMARY:
        d = BOOT[PRIMARY] - BOOT[n]
        GAIN[n] = dict(d_auc=RESULTS.loc[PRIMARY, "auc"] - RESULTS.loc[n, "auc"], lo=np.percentile(d, 2.5),
                       hi=np.percentile(d, 97.5), p=min(1.0, 2 * min((d <= 0).mean(), (d >= 0).mean())))
GAIN = pd.DataFrame(GAIN).T
pd.set_option("display.width", 200)
print("C3, AD vs FTD, 59 patients, 5 x 5-fold; mean over repeats\n")
print(RESULTS.round(3).to_string())
print("\naveraged score minus each recording alone (paired bootstrap)\n")
print(GAIN.round(3).to_string())


C3, AD vs FTD, 59 patients, 5 x 5-fold; mean over repeats

                                            auc  auc_lo  auc_hi  recall_AD  recall_FTD  bal_acc
run                                                                                            
ds004504 eyes closed, 300 s               0.628   0.502   0.748      0.722       0.487    0.605
ds006036 eyes open, 44 s (Experiment 10)  0.641   0.515   0.753      0.656       0.539    0.597
both recordings, averaged (primary)       0.658   0.527   0.773      0.739       0.513    0.626

averaged score minus each recording alone (paired bootstrap)

                                          d_auc     lo     hi      p
ds004504 eyes closed, 300 s               0.030 -0.048  0.101  0.450
ds006036 eyes open, 44 s (Experiment 10)  0.017 -0.052  0.087  0.594


In [6]:
# ===========================================================================
# CELL 6 — PRIMARY TEST: BOTH PIPELINES ON THE SAME SHUFFLED DIAGNOSES
# ===========================================================================
def shuffled_auc(k):
    labels = LABELS.copy()
    labels[DEMENTIA] = np.random.default_rng(SEED + 20_000 + k).permutation(labels[DEMENTIA])
    q = ((run_cv(X04, labels) + run_cv(X06, labels)) / 2)[:, DEMENTIA]
    yd = labels[DEMENTIA] == AD
    return auc(q[:, yd], q[:, ~yd]).mean()


t0 = time.time()
NULL = np.array(Parallel(n_jobs=N_JOBS)(delayed(shuffled_auc)(k) for k in range(N_PERM)))
OBS = RESULTS.loc[PRIMARY, "auc"]
P_PERM = (1 + (NULL >= OBS).sum()) / (1 + len(NULL))
print(f"{len(NULL)} shuffles in {time.time() - t0:.0f} s")
print(f"averaged C3 AUC {OBS:.3f}; shuffled: median {np.median(NULL):.3f}, 95 % range "
      f"{np.percentile(NULL, 2.5):.3f}-{np.percentile(NULL, 97.5):.3f}; one-sided p = {P_PERM:.4f}")
ADOPT = bool(GAIN.loc["ds006036 eyes open, 44 s (Experiment 10)", "lo"] > 0)
print(f"\nadopted as stage 2 (pre-registered rule: gain over ds006036 alone, lower 95 % bound > 0): {ADOPT}")


1000 shuffles in 1303 s
averaged C3 AUC 0.658; shuffled: median 0.481, 95 % range 0.323-0.643; one-sided p = 0.0210

adopted as stage 2 (pre-registered rule: gain over ds006036 alone, lower 95 % bound > 0): False


In [7]:
# ===========================================================================
# CELL 7 — SECONDARY: SEVERITY AND THE TWO-STAGE 3-CLASS CHECK
# ===========================================================================
_s = subjects.iloc[DEMENTIA]


def loglik(Xc, y):
    m = LogisticRegression(C=np.inf, max_iter=10000).fit(Xc, y)
    p = np.clip(m.predict_proba(Xc)[:, 1], 1e-12, 1 - 1e-12)
    return (y * np.log(p) + (1 - y) * np.log(1 - p)).sum()


_base = StandardScaler().fit_transform(np.column_stack([_s.mmse, _s.age, _s.sex == "F"]).astype(float))
_ll0 = loglik(_base, YD)
conf = []
for n, q in Q.items():
    qm = np.clip(q[:, DEMENTIA].mean(0), 1e-6, 1 - 1e-6)
    lq = np.log(qm / (1 - qm))
    lr = 2 * (loglik(np.column_stack([_base, (lq - lq.mean()) / lq.std()]), YD) - _ll0)
    conf.append(dict(run=n, lr_chi2=lr, p_beyond_mmse_age_sex=stats.chi2.sf(lr, 1),
                     r_with_mmse=stats.spearmanr(qm, _s.mmse)[0]))
CONFOUNDS = pd.DataFrame(conf).set_index("run")
print("does each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)\n")
print(CONFOUNDS.round(3).to_string())

# ---- two-stage 3-class: the soft vote's P(dementia) x q, repeats 1-3 (the 3-class notebook's folds) ----
_n = [len(np.load(PROJECT / "subject_cache" / f"sub-{i:03d}.npz")["y"]) for i in range(1, 89)]
_sid = np.repeat(np.arange(88), _n)
_ens = np.load(PROJECT / "results_gat_multiclass" / "ensemble_oof_prob.npy")          # (3, windows, 3)
_S = np.stack([np.stack([_ens[r][_sid == s].mean(0) for s in range(88)]) for r in range(len(_ens))])
three = []
for n, q in [("none (today's soft vote)", None)] + list(Q.items()):
    m = []
    for r in range(len(_S)):
        P = _S[r]
        if q is not None:
            d = 1 - P[:, HC]
            P = np.column_stack([1 - d, d * q[r], d * (1 - q[r])])
        pred = P.argmax(1)
        rec = recall_score(LABELS, pred, labels=[HC, AD, FTD], average=None)
        m.append(dict(acc=accuracy_score(LABELS, pred), macro_f1=f1_score(LABELS, pred, average="macro"),
                      macro_recall=rec.mean(), recall_HC=rec[0], recall_AD=rec[1], recall_FTD=rec[2]))
    three.append(dict(stage_2=n, **pd.DataFrame(m).mean().to_dict()))
THREE = pd.DataFrame(three).set_index("stage_2")
print("\ntwo-stage 3-class, stage 1 = GAT + band-power soft vote, mean of repeats 1-3\n")
print(THREE.round(3).to_string())


does each score add to MMSE + age + sex? (likelihood-ratio test, 1 df)

                                          lr_chi2  p_beyond_mmse_age_sex  r_with_mmse
run                                                                                  
ds004504 eyes closed, 300 s                 0.460                  0.498       -0.361
ds006036 eyes open, 44 s (Experiment 10)    1.782                  0.182       -0.274
both recordings, averaged (primary)         1.699                  0.192       -0.356

two-stage 3-class, stage 1 = GAT + band-power soft vote, mean of repeats 1-3

                                            acc  macro_f1  macro_recall  recall_HC  recall_AD  recall_FTD
stage_2                                                                                                  
none (today's soft vote)                  0.625     0.592         0.600      0.782      0.685       0.333
ds004504 eyes closed, 300 s               0.648     0.633         0.635      0.770      0.657       0

In [8]:
# ===========================================================================
# CELL 8 — SAVE
# ===========================================================================
np.savez_compressed(OUT_DIR / "c3_both_oof.npz", subjects=subjects.participant_id.to_numpy(), labels=LABELS,
                    runs=np.array(list(Q)), q_ad=np.stack([Q[n] for n in Q]), fold_of=FOLD_OF)
RESULTS.to_csv(OUT_DIR / "results.csv")
GAIN.to_csv(OUT_DIR / "gain.csv")
CONFOUNDS.to_csv(OUT_DIR / "confounds.csv")
THREE.to_csv(OUT_DIR / "three_class.csv")
pd.DataFrame(dict(shuffled_auc=NULL)).to_csv(OUT_DIR / "permutation.csv", index=False)
_p = RESULTS.loc[PRIMARY]
SUMMARY = dict(task="AD vs FTD, 36 vs 23: C3 averaged over both recordings of each patient",
               protocol=PROTOCOL["written"],
               primary=dict(auc=round(_p.auc, 4), ci=[round(_p.auc_lo, 4), round(_p.auc_hi, 4)],
                            p_permutation=round(P_PERM, 4), n_perm=N_PERM, adopted_as_stage_2=ADOPT),
               gain={n: {k: round(float(v), 4) for k, v in r.items()} for n, r in GAIN.iterrows()},
               p_beyond_mmse_age_sex=round(CONFOUNDS.loc[PRIMARY, "p_beyond_mmse_age_sex"], 4),
               three_class={n: {k: round(float(v), 4) for k, v in r.items()} for n, r in THREE.iterrows()})
(OUT_DIR / "summary.json").write_text(json.dumps(SUMMARY, indent=1, ensure_ascii=False, default=float),
                                      encoding="utf-8")
print(f"saved to {OUT_DIR}:", ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))


saved to C:\Projects\Major Project\experiments_ad_vs_ftd\results\c3_both: c3_both_oof.npz, confounds.csv, gain.csv, permutation.csv, protocol.json, results.csv, summary.json, three_class.csv
